# Hands-On Exercise with Tabular AI orchestration

This notebook illustrates how you can use Tabular AI orchestration to run predictions at scale. 

## Setup

Import all required packages and load in the environment variables from the `.env` file.

In [ ]:
from dotenv import load_dotenv

import requests
import json
import pandas as pd
import urllib3
import os


In [ ]:
load_dotenv()

# AI Core credentials - used to retrieve the bearer token for subsequent API calls to set up and run the orchestration workflow
AUTH_URL = os.getenv("AICORE_AUTH_URL")
CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
AI_API_URL = os.getenv("AICORE_BASE_URL")

# HANA Data Lake File System details - if you are on your own set up, you can change this to your own HDLFS instance
HDLFS_URL = os.getenv("TABORC_HDLFS_URL")
HDLFS_HOST = os.getenv("TABORC_HDLFS_HOST")
HDLFS_PATH = os.getenv("TABORC_HDLFS_PATH")


In [ ]:
def print_response(res):
    """Pretty-print an HTTP response for inspection."""
    print(f"Status: {res.status_code} {res.reason}")
    try:
        print(json.dumps(res.json(), indent=2))
    except Exception:
        print(res.text)


---
## Exercise 1.1 - Setting up Tabular AI orchestration

There are 4 one-time setup steps, in the following order:
1. **Create a Data Destination** - register the HDLFS connection in the Context Registry
2. **Create a Tabular Artifact** - register the provided dataset with its schema (the Context Registry validates the file exists at registration time)
3. **Create a Scenario Configuration** - create the scenario that the workflow can refer to at runtime to retrive the registered tabular artifact
4. **Deploy Tabular Orchestration** - create a virtual deployment so that you can trigger the workflow at runtime **(NOTE: You do not need to run this step for this exercise)**

In [ ]:
# ── TODO: Set your personal suffix once and keep it for the whole training ──────────
# Use something short and unique to you - your initials or first name works fine.
# This keeps your Data Destination, Artifact, and Scenario Config stable across
# sessions. Changing it creates new resources and leaves the old ones orphaned.
# ─────────────────────────────────────────────────────────────────────────────
SUFFIX = ""   # e.g. "alex" or "js" - set once, never change

print(f"Your suffix    : {SUFFIX}")
print(f"Resource group : {RESOURCE_GROUP}")
print(f"\nYour resources will be named:")
print(f"  Data Destination  : dd-cars-{SUFFIX}")
print(f"  Tabular Artifact  : ta-cars-{SUFFIX}")
print(f"  Scenario Config   : cars-scenario-{SUFFIX}")


In [ ]:
response = requests.get(
    AUTH_URL + "/oauth/token?grant_type=client_credentials",
    auth=(CLIENT_ID, CLIENT_SECRET)
).json()

BEARER_TOKEN = response["access_token"]

# AI Core headers required by all Context Registry calls
HEADERS = {
    "AI-Resource-Group": RESOURCE_GROUP,
    "Content-Type": "application/json",
    "Authorization" : f"Bearer {BEARER_TOKEN}"
}

if BEARER_TOKEN != "":
    print("Token fetched successfully.")


### Step 1: Register a Data Destination

A **Data Destination** tells the Context Registry where your data lives and how to connect to it. This is separate from the upload, because it registers the credentials, not the file itself.

> **Note:** For this hands-on exercise, all participants will share the same HANA Date Lake Files instance and dataset.

In [ ]:
CONTEXT_REGISTRY_URL = f"{AI_API_URL}/v2/tcr"

DATA_DESTINATION_NAME = f"dd-cars-{SUFFIX}"

payload = {
    "type": "HDL",
    "description": f"Cars context data - {SUFFIX}",
    "adapterType": "File",
    "config": {
        "host": HDLFS_URL 
    },
}

DATA_DESTINATION_URL = f"{CONTEXT_REGISTRY_URL}/dataDestinations/{DATA_DESTINATION_NAME}"

print(f"Creating a data destination via {DATA_DESTINATION_URL}")
response = requests.put(DATA_DESTINATION_URL, headers=HEADERS, json=payload)
print_response(response)
# On success: 201 Created

### Step 2: Upload to HDLFS and Register a Tabular Artifact

For your convenience, we have already uploaded the datset to HANA Data Lake Files. You can use the provided file link to register the Tabular Artifact pointing to that file. The Context Registry validates the file exists at registration time, so the upload must succeed first.

In [ ]:
TABULAR_ARTIFACT_NAME = f"ta-cars-{SUFFIX}"

# Column names must match the uploaded CSV exactly.
# selectedColumns tells the Context Registry which columns to index.
# The CSN definition maps each column to a type so RPT knows how to embed them.
# NOTE: every column present in the CSV header must be defined here, otherwise
# the Context Registry CSV parser fails on the header line ("parsing failed on line 0").
CARS_COLUMNS = [
    "idx",
    "Car Brand",
    "Model",
    "Price",
    "Model Year",
    "Location",
    "Fuel",
    "Driven (Kms)",
    "Gear",
    "Ownership"
]

payload = {
    "dataDestinationName": DATA_DESTINATION_NAME,
    "type": "CSV",
    "path": f"{HDLFS_PATH}",
    "csnMetadata": {
        "entityName": "used-cars",
        "selectedColumns": CARS_COLUMNS,
        "definition": {
            "definitionType": "DOCUMENT",
            "document": {
                "definitions": {
                    "used-cars": {
                        "kind": "entity",
                        "elements": {
                            "idx":                {"type": "cds.Integer"},
                            "Car Brand":          {"type": "cds.String"},
                            "Model":              {"type": "cds.String"},
                            "Price":              {"type": "cds.Integer"},
                            "Model Year":         {"type": "cds.Integer"},
                            "Location":           {"type": "cds.String"},
                            "Fuel":               {"type": "cds.String"},
                            "Driven (Kms)":       {"type": "cds.Integer"},
                            "Gear":               {"type": "cds.String"},
                            "Ownership":          {"type": "cds.Integer"}
                        }
                    }
                }
            }
        }
    }
}

TABULAR_ARTIFACT_URL = f"{CONTEXT_REGISTRY_URL}/tabularArtifacts/{TABULAR_ARTIFACT_NAME}"

print(f"Creating a tabular artifact via {TABULAR_ARTIFACT_URL}")
response = requests.put(TABULAR_ARTIFACT_URL, headers=HEADERS, json=payload)
print_response(response)
# On success: 201 Created

### Step 3: Create a Scenario Configuration

A **Scenario Configuration** defines the scenario that the orchestration workflow will refer to at runtime. At prediction time, you pass `scenarioConfigName` to the Orchestration workflow — it looks up which Tabular Artifact to pull context from automatically.

> **Note:** It might take a few minutes before your created Tabular Artifact turns to the `ACTIVE` state. If you get an error while creating the scenario configuration, wait a short while before you run the following code.

In [ ]:
SCENARIO_CONFIG_NAME = f"cars-scenario-{SUFFIX}"

payload = {
    "tabularArtifacts": [
        {"name": TABULAR_ARTIFACT_NAME}
    ],
}

SCENARIO_CONFIG_URL = f"{CONTEXT_REGISTRY_URL}/scenarioConfigurations/{SCENARIO_CONFIG_NAME}"

print(f"Creating a scenario configuration via {TABULAR_ARTIFACT_URL}")
response = requests.put(SCENARIO_CONFIG_URL, headers=HEADERS, json=payload)
print_response(response)
# On success: 201 Created
print(f"\nYour scenario config name: {SCENARIO_CONFIG_NAME}")


### Step 4: Create a Deployment for Tabular Orchestration 

> **IMPORTANT**: Tabular AI orchestration has already been deployed for you. Simply use the provided deployment ID to start running predictions. DO NOT RUN THE COMMENTED CODE.

If you are doing this for the very first time on you own setup, you will need to deploy Tabular AI orchestration and here are 2 sub-steps:
1. Create a configuration using the `tabular-orchestration` executable and scenario
2. Create the deployment using the configuration ID from the previous step

**Note:** You will see an `UNKNOWN` status immediately after creating the deployment. It might take a few minutes before it turns to the `RUNNING` ACTIVE state. You can run the next code block to check on the deployment status.

In [ ]:
# DO NOT RUN THIS PIECE OF CODE

# # Do not change these variables if you are trying out on your own
# ORCHESTRATION_EXECUTABLE = "tabular-orchestration"
# ORCHESTRATION_SCENARIO = "tabular-orchestration"

# ORCHESTRATION_CONFIG_NAME = f"tab-orc-{SUFFIX}"

# # Create the configuration for the workflow
# payload = {
#     "name": ORCHESTRATION_CONFIG_NAME,
#     "executableId": ORCHESTRATION_EXECUTABLE,
#     "scenarioId": ORCHESTRATION_SCENARIO,
#     "parameterBindings": []
# }

# response = requests.post(AI_API_URL + "/v2/lm/configurations", headers=HEADERS, json=payload)
# print_response(response)
# # On success: 201 Created
# ORCHESTRATION_CONFIG_ID = (response.json())["id"]
# print(f"\nConfig ID for Tabular Orchestration: {ORCHESTRATION_CONFIG_ID}")

# # Create the deployment using the configuration ID
# payload = {
#     "configurationId": ORCHESTRATION_CONFIG_ID
# }

# response = requests.post(AI_API_URL + "/v2/lm/deployments", headers=HEADERS, json=payload)
# print_response(response)
# # On success: 201 Created
# ORCHESTRATION_DEPLOYMENT_ID = (response.json())["id"]
# print(f"\nUse this deployment ID to run the Orchestration workflow: {ORCHESTRATION_DEPLOYMENT_ID}")

In [ ]:
ORCHESTRATION_DEPLOYMENT_ID = os.getenv("TABORC_DEPLOYMENT_ID")

# Check the status of the created deployment
# It should be in RUNNING state before you proceed
response = requests.get(AI_API_URL + "/v2/lm/deployments/" + ORCHESTRATION_DEPLOYMENT_ID, headers=HEADERS)
print_response(response)


### Summary for Exercise 3.1

```
Dataset on HDLFS  ->  Data Destination  ->  Tabular Artifact  ->  Scenario Config  ->  Tabular Orchestration Deployment
  (your data)         (where it lives)       (what it is)         (runtime handle)          (virtual deployment)
```

At runtime (Step 3), your application will need the Orchestration URL (specified with the deployment ID) and `scenarioConfigName`. Everything else is handled automatically.

---
## Exercise 1.2: Live Prediction via Tabular AI orchestration

With the Tabular AI orchestration configured in Exercise 3.1, we can now make predictions using the configured workflow. Instead of manually bundling context rows into every request, we can simply send the query rows — the Orchestration workflow will **automatically** fetch the context from the registered Tabular Artifact.

```
Your request (query rows only)
         |
         v
Tabular AI Orchestrator
    |              |
    v              v
Context Selector   SAP RPT
    |
Tabular Artifact (registered in Step 2 of Exercise 1.1)
```

In [ ]:
print(f"Orchestration Deployment ID : {ORCHESTRATION_DEPLOYMENT_ID}")
print(f"Scenario config             : {SCENARIO_CONFIG_NAME}")
print(f"Resource group              : {RESOURCE_GROUP}")

ORCHESTRATION_URL = f"{AI_API_URL}/v2/inference/deployments/{ORCHESTRATION_DEPLOYMENT_ID}/predict"
print(f"Orchestration URL           : {ORCHESTRATION_URL}")

# CLIENT_ID, CLIENT_SECRET, and AUTH_URL should already copied to the .env file and retrieved during setup
# Re-run that cell if you get auth errors here.

ORCHESTRATION_HEADERS = {
    "Authorization": f"Bearer {BEARER_TOKEN}",
    "ai-resource-group": RESOURCE_GROUP,
    "Content-Type": "application/json",
}

In [ ]:
# Extract the query rows
QUERY_ROWS_CSV_PATH = "./data/used_cars_24.csv"

query_rows = pd.read_csv(QUERY_ROWS_CSV_PATH)

assert len(query_rows) > 0, "No rows loaded from CSV file!"
print(f"Loaded query rows: {len(query_rows)}")
print(f"Columns: {query_rows.columns.tolist()}")

# Replace target column with [PREDICT] placeholder
REGRESSION_TARGET_COLUMN = "Price"
CLASSIFICAION_TARGET_COLUMN = "Ownership"
PREDICT_PLACEHOLDER = "[PREDICT]"
query_rows_processed = query_rows.copy()
query_rows_processed[REGRESSION_TARGET_COLUMN] = PREDICT_PLACEHOLDER
query_rows_processed[CLASSIFICAION_TARGET_COLUMN] = PREDICT_PLACEHOLDER
query_rows_processed.head()

In [ ]:
# feel free to change up the values in these variables
MODEL_NAME = "sap-rpt-1.5" # we also support `sap-rpt-1-small`, `sap-rpt-1-large`, `sap-rpt-1.5` and `sap-rpt-1.5-large`
CONTEXT_SELECTION_STRATEGY = "random" # we also support "heuristic", a more advanced selection approach for improved prediction accuracy
NUM_CONTEXT_ROWS = 2048 # you can increase this if you want to use the large variants
TOP_K = 2 # we support up to 20
NUM_QUERY_ROWS = 128 # we support up to 128 query rows


orchestration_payload = {
    # TODO: update to sap-rpt-1.6 once available
    "modelName": MODEL_NAME,
    "scenarioConfigName": SCENARIO_CONFIG_NAME,
    "contextSelectionConfig": {
        "numRows": NUM_CONTEXT_ROWS,
        "indexColumn": "idx",
        "strategy": CONTEXT_SELECTION_STRATEGY,
    },
    "predictionConfig": {
        "targetColumns": [
          {
            "name": REGRESSION_TARGET_COLUMN,
            "predictionPlaceholder": PREDICT_PLACEHOLDER,
            "taskType": "regression"
          },
          {
            "name": CLASSIFICAION_TARGET_COLUMN,
            "predictionPlaceholder": PREDICT_PLACEHOLDER,
            "taskType": "classification",
            # "topK": TOP_K
          }
        ]
    },
      "rows": query_rows_processed.head(NUM_QUERY_ROWS).fillna("").to_dict(orient="records"),
}

print(json.dumps(orchestration_payload, indent=2))

In [ ]:
response = requests.post(ORCHESTRATION_URL, headers=ORCHESTRATION_HEADERS, json=orchestration_payload)
print(f"Status: {response.status_code}")

if response.status_code != 200:
    raise RuntimeError(f"Orchestration error {response.status_code}: {response.text}")

result = response.json()
predictions = result["predictions"]
# On success, status = 200
print(json.dumps(predictions, indent=2))

---
## Summary

This notebook covered the full Tabular Orchestration workflow stack:

```
Tabular Orchestration Setup (Exercise 3.1)  ->  Prediction (Exercise 3.2)
```

A few natural next steps from here:
- **Try it on your use case**: The entire workflow should remain largely the same. Replace our example datasets with your own, change up the data preparation logic to what suits your dataset and redo the orchestration configuration.

- **Compare between models**: Change `MODEL_NAME` in the payload cell and re-run the predictions. We support `sap-rpt-1` variants and `sap-rpt-1.5` variants as well.

- **Scale up**: Switch to `sap-rpt-1.6-large` and increase `numRows` in `contextSelectionConfig` for higher accuracy and confidence intervals on predictions.

- **Try top-k predictions**: Set `"topK": 2` in `targetColumns` to get ranked predictions.